(fides_tutorial)=

# Getting the most out of the fides optimizer

[Fides](https://fides-optimizer.readthedocs.io) is a trust-region optimizer for smooth
functions with or without bounds. It needs gradients and typically needs few function
evaluations, which makes it a good choice for expensive functions.

Fides works out of the box, but it is also one of the most configurable optimizers
in optimagic. In this tutorial you learn:

- how to use fides with its defaults
- how to keep fides away from regions where your function is not defined
- how to choose and customize the Hessian approximation
- which options matter for bounds and for problems with many parameters
- how to control when fides stops

To run the tutorial you need fides (`pip install "fides>=0.8.2"`).

In [ ]:
import time

import numpy as np
import pandas as pd

import optimagic as om

## Fides with default options

We start with the Rosenbrock function, a standard test problem with a curved valley
that leads to the optimum at `[1, 1, ..., 1]`. Fides evaluates the function and its
gradient together in every iteration, so we provide both in one function via
`fun_and_jac`.

In [ ]:
def rosenbrock(x):
    return (100 * (x[1:] - x[:-1] ** 2) ** 2 + (1 - x[:-1]) ** 2).sum()


def rosenbrock_and_gradient(x):
    grad = np.zeros_like(x)
    grad[:-1] = -400 * x[:-1] * (x[1:] - x[:-1] ** 2) - 2 * (1 - x[:-1])
    grad[1:] += 200 * (x[1:] - x[:-1] ** 2)
    return rosenbrock(x), grad


res = om.minimize(
    fun=rosenbrock,
    fun_and_jac=rosenbrock_and_gradient,
    params=np.full(5, -1.0),
    algorithm="fides",
)
res.params.round(5), res.n_iterations, res.message

The message tells you why fides stopped. Here the gradient is close enough to zero,
which means that the parameters are close to a local optimum.

All options of fides can be set when you select the algorithm, either as
`om.algos.fides(...)` or via `algo_options`. The following sections show the options
that are most useful in practice. You find the complete list in the
[algorithm documentation](fides-algorithm).

## Staying inside the region where the function is defined

Many functions are not defined for all parameter values, and often the region where
they are defined cannot be described by simple bounds. An optimizer that takes a long
first step can leave this region before it has learned anything about the function.

As an example we fit a generalized Pareto distribution to data, as it is done for
insurance losses or flood heights. The distribution has a `scale` and a `shape`
parameter. Its likelihood is only defined if `1 + shape * y / scale > 0` for every
observation `y`. Whether parameters are valid therefore depends on the data.

In [ ]:
rng = np.random.default_rng(0)
uniform = rng.uniform(size=500)
data = 2.0 / 0.2 * ((1 - uniform) ** (-0.2) - 1)  # scale = 2, shape = 0.2


def pareto_negative_loglike_and_gradient(params):
    scale, shape = params
    z = 1 + shape * data / scale
    if scale <= 0 or (z <= 0).any():
        raise ValueError("The likelihood is not defined at these parameters.")
    n_obs = len(data)
    value = n_obs * np.log(scale) + (1 + 1 / shape) * np.log(z).sum()
    grad_scale = n_obs / scale - (1 + 1 / shape) * (shape * data / scale**2 / z).sum()
    grad_shape = (
        -np.log(z).sum() / shape**2 + (1 + 1 / shape) * (data / scale / z).sum()
    )
    return value, np.array([grad_scale, grad_shape])


def pareto_negative_loglike(params):
    return pareto_negative_loglike_and_gradient(params)[0]


def fit_pareto(algorithm):
    return om.minimize(
        fun=pareto_negative_loglike,
        fun_and_jac=pareto_negative_loglike_and_gradient,
        params=np.array([4.0, 0.1]),
        algorithm=algorithm,
    )

With default options, fides fails. So does L-BFGS-B from scipy. Both step into the
undefined region right at the start:

In [ ]:
for algorithm in [om.algos.fides, om.algos.scipy_lbfgsb]:
    try:
        fit_pareto(algorithm)
    except Exception as error:
        print(f"{algorithm.name} failed: {str(error).splitlines()[0]}")

The reason is the length of the first step. Fides only trusts its model of the
function inside the trust region, and the first step can be as long as the initial
radius of this region. The default radius of 1 is too long here: The gradient at the
start parameters is large and points towards invalid parameters.

`trustregion_initial_radius` makes the first step shorter. Afterwards fides adapts the
radius on its own: It grows when the model predicts the function well and shrinks
otherwise.

In [ ]:
res = fit_pareto(om.algos.fides(trustregion_initial_radius=0.1))
res.params.round(3), res.n_iterations

Fides now converges in a few iterations. The estimates are close to the values that
generated the data (2 and 0.2).

```{note}
Choose the initial radius in the units of your parameters: It is the largest distance
the parameters can move in the first iteration. If a short first step is not enough,
you can also let the radius grow more slowly with `trustregion_expansion_factor`.
A small radius does not guarantee that the function is never evaluated at invalid
parameters. For this case, optimagic can replace failed evaluations by a penalty; see
[how to handle errors](../how_to/how_to_errors_during_optimization.ipynb).
```

## Choosing the Hessian approximation

Fides builds a quadratic model of the function in every iteration. For this it needs
the Hessian, which it approximates from the gradients it has seen. The approximation
decides how good the steps are, so `hessian_update` is the option with the largest
effect on the number of iterations.

The two most useful choices are `"bfgs"` (the default) and `"sr1"`. BFGS keeps the
model convex, which is robust. SR1 can also represent negative curvature. Which one is
better depends on the problem, and trying both is cheap. We compare them on the
Rosenbrock function and on the Mancino function from optimagic's benchmark problems.

In [ ]:
mancino = om.get_benchmark_problems("more_wild")["mancino_5_good_start"]["inputs"]

comparison = {}
for hessian_update in ["bfgs", "sr1"]:
    algorithm = om.algos.fides(hessian_update=hessian_update)
    res_rosenbrock = om.minimize(
        fun=rosenbrock,
        fun_and_jac=rosenbrock_and_gradient,
        params=np.full(5, -1.0),
        algorithm=algorithm,
    )
    res_mancino = om.minimize(
        fun=mancino["fun"],
        params=mancino["params"],
        algorithm=algorithm,
        numdiff_options=om.NumdiffOptions(method="central"),
    )
    comparison[hessian_update] = {
        "rosenbrock": res_rosenbrock.n_iterations,
        "mancino": res_mancino.n_iterations,
    }

pd.DataFrame(comparison).rename_axis("iterations until convergence")

BFGS is clearly better on the Rosenbrock function, SR1 is clearly better on the
Mancino function. If your function is expensive, such a comparison on a small version
of your problem pays off quickly.

### Customizing the approximation with objects from fides

The names are shortcuts. For full control you can pass an object from
`fides.hessian_approximation` instead. Two examples:

- `Broyden(phi=...)` mixes the BFGS update (`phi=0`) and the DFP update (`phi=1`).
- `BFGS(enforce_curv_cond=False)` also updates the approximation after steps where
  the gradients indicate negative curvature. By default such updates are skipped.

In [ ]:
from fides.hessian_approximation import BFGS, Broyden

custom_updates = {
    "bfgs": "bfgs",
    "Broyden(phi=0.5)": Broyden(phi=0.5),
    "BFGS(enforce_curv_cond=False)": BFGS(enforce_curv_cond=False),
}

iterations = {}
for name, hessian_update in custom_updates.items():
    res = om.minimize(
        fun=mancino["fun"],
        params=mancino["params"],
        algorithm=om.algos.fides(hessian_update=hessian_update),
        numdiff_options=om.NumdiffOptions(method="central"),
    )
    iterations[name] = res.n_iterations

pd.Series(iterations, name="iterations on the Mancino function")

```{note}
Fides also has Hessian approximations that need exact Hessians (`HybridFixed`,
`HybridFraction`) or least-squares residuals (`FX`, `SSM`, `TSSM`, `GNSBFGS`). They
are not supported in optimagic yet.
```

## Problems with bounds

Fides keeps the parameters strictly inside the bounds. Two options control what
happens when a step would cross a bound:

- `stepback_strategy` decides how the step is changed. With `"reflect"` the step
  bounces off the bound, with `"truncate"` it stops there and the remaining
  parameters are optimized again. `"mixed"` and `"refine"` invest more computation to
  find a better step.
- `max_stepback_fraction` decides how close a single step may get to a bound.

The differences are usually small, as in the following example where an upper bound
of 0.5 cuts off the optimum of the Rosenbrock function. Trying the strategies is
most promising if many parameters end up at a bound and fides needs many
iterations.

In [ ]:
n_params = 50
bounds = om.Bounds(lower=np.full(n_params, -2.0), upper=np.full(n_params, 0.5))

iterations = {}
for strategy in ["reflect", "reflect_single", "truncate", "mixed", "refine"]:
    res = om.minimize(
        fun=rosenbrock,
        fun_and_jac=rosenbrock_and_gradient,
        params=np.full(n_params, -1.0),
        bounds=bounds,
        algorithm=om.algos.fides(stepback_strategy=strategy),
    )
    iterations[strategy] = res.n_iterations

pd.Series(iterations, name="iterations")

```{note}
If you do not set `stepback_strategy` on a problem with bounds, optimagic currently
uses `"truncate"` and warns that the default will change to `"reflect"`, the default
of fides, in optimagic 0.7.0. Set the option explicitly to silence the warning.
```

## Problems with many parameters

In every iteration fides has to find the best step for its quadratic model. How this
is done is controlled by `subspace_solver`:

- `"full"` finds the exact solution. This is the default in optimagic.
- `"2D"` only searches a two-dimensional plane. This is the default of fides.
- `"scg"` uses a conjugate gradient method that stops early.

For expensive functions the number of iterations is what counts. For cheap functions
with many parameters, the time spent inside the optimizer becomes important. `"scg"`
needs more iterations, but each one is much cheaper:

In [ ]:
rows = {}
for solver in ["full", "2D", "scg"]:
    start = time.perf_counter()
    res = om.minimize(
        fun=rosenbrock,
        fun_and_jac=rosenbrock_and_gradient,
        params=np.full(100, -1.0),
        algorithm=om.algos.fides(subspace_solver=solver),
    )
    rows[solver] = {
        "iterations": res.n_iterations,
        "seconds": round(time.perf_counter() - start, 2),
    }

pd.DataFrame.from_dict(rows, orient="index")

With 100 parameters and a function that takes microseconds, `"scg"` is the fastest
although it needs about three times as many iterations. If one evaluation of your
function takes a second, the picture is reversed and `"full"` is the better choice.

## Controlling when fides stops

Fides has two kinds of convergence criteria:

- The gradient criteria `convergence_gtol_abs` and `convergence_gtol_rel` measure
  closeness to an optimum: At an optimum the gradient is zero. These are the main
  criteria.
- `convergence_ftol_abs`, `convergence_ftol_rel` and `convergence_xtol_abs` measure
  progress: Fides stops when the function value or the parameters hardly change
  anymore. They are a safety net for problems where the gradient cannot become small
  enough, for example because it is noisy.

The defaults in optimagic are stricter than the defaults of fides, so that most runs
are ended by a gradient criterion. You see which criterion ended a run in
`res.message`.

If you need a more precise solution, tighten the gradient tolerance. If function
evaluations are expensive and a rough solution is enough, loosen it:

In [ ]:
rows = {}
for gtol in [1e-2, 1e-5, 1e-8]:
    res = om.minimize(
        fun=rosenbrock,
        fun_and_jac=rosenbrock_and_gradient,
        params=np.full(5, -1.0),
        algorithm=om.algos.fides(convergence_gtol_abs=gtol),
    )
    rows[gtol] = {
        "iterations": res.n_iterations,
        "distance to optimum": np.abs(res.params - 1).max(),
    }

pd.DataFrame.from_dict(rows, orient="index").rename_axis("convergence_gtol_abs")

You can also limit the resources of a run with `stopping_maxiter` and
`stopping_maxtime` (in seconds). A run that hits one of these limits is reported as
not successful:

In [ ]:
res = om.minimize(
    fun=rosenbrock,
    fun_and_jac=rosenbrock_and_gradient,
    params=np.full(5, -1.0),
    algorithm=om.algos.fides(stopping_maxiter=10),
)
res.success, res.message

## Fine-tuning the trust region

After every step fides compares the actual improvement of the function with the
improvement its model predicted. Four options control how the radius of the trust
region reacts:

| Option | Default | Effect |
| --- | --- | --- |
| `trustregion_shrinking_threshold` | 0.25 | The radius shrinks if the step achieved at most this share of the predicted improvement. |
| `trustregion_shrinking_factor` | 0.25 | The radius is multiplied by this factor when it shrinks. |
| `trustregion_expansion_threshold` | 0.75 | The radius can grow if the step achieved at least this share of the predicted improvement. |
| `trustregion_expansion_factor` | 2.0 | The radius is multiplied by this factor when it grows. |

The defaults work well for most problems. Changing them is worth a try in two
situations:

- The function has regions where it is not defined or changes abruptly. A smaller
  expansion factor makes fides more careful after a good step.
- The start parameters are far from the optimum and the function is well behaved. A
  larger expansion factor gets fides up to speed in fewer iterations.

The following configuration is a cautious one for the generalized Pareto example
from above:

In [ ]:
cautious = om.algos.fides(
    trustregion_initial_radius=0.1,
    trustregion_expansion_factor=1.5,
    trustregion_shrinking_factor=0.1,
)
res = fit_pareto(cautious)
res.params.round(3), res.n_iterations

## Summary

| Situation | Option to try |
| --- | --- |
| The first steps leave the region where the function is defined | `trustregion_initial_radius`, `trustregion_expansion_factor` |
| Fides needs many iterations | `hessian_update="sr1"` or a customized object from fides |
| Many parameters are at a bound | `stepback_strategy` |
| Many parameters and a cheap function | `subspace_solver="scg"` |
| The solution is not precise enough | `convergence_gtol_abs` |
| The gradient is noisy and fides does not stop | `convergence_ftol_rel` |